# Data Cleaning: Seattle Airbnb Listings

## Overview
This notebook cleans and prepares Seattle Airbnb listings data for analysis. It contains my data preparation work from a larger group project and focuses on creating a consistent, comparable sample for neighborhood level pricing analysis.

The main cleaning steps are:

- restrict listings to Seattle neighborhoods included in the neighborhood reference file
- remove columns that are not needed for pricing analysis
- convert key variables to usable data types
- backfill missing prices using prior quarter data where available
- remove remaining missing prices and extreme price outliers
- filter to comparable short-term listings
- export the cleaned dataset for analysis


## 1. Import Libraries and Load Data

The analysis uses third quarter 2025 Seattle listings as the main dataset. Second quarter 2025 listings are used only to backfill missing prices where the same listing appears in both quarters.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

# Main listings file for analysis
listings = pd.read_csv("../data/listings_3Q25.csv")

# Prior quarter listings used only for missing price backfill
listings_prior = pd.read_csv("../data/listings_2Q25.csv")

# Neighborhood reference file
neighborhoods = pd.read_csv("../data/neighbourhoods.csv")

print(f"Listings shape: {listings.shape}")
print(f"Prior quarter listings shape: {listings_prior.shape}")
print(f"Neighborhood reference shape: {neighborhoods.shape}")

listings.head()


Listings shape: (6996, 79)
Prior quarter listings shape: (6862, 79)
Neighborhood reference shape: (90, 2)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6606,https://www.airbnb.com/rooms/6606,20250925032813,2025-09-25,city scrape,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","A peaceful yet highly accessible neighborhood,...",https://a0.muscache.com/pictures/45742/21116d7...,14942,https://www.airbnb.com/users/show/14942,Joyce,2009-04-26,"Seattle, WA",I am a therapist/innkeeper.I know my city well...,within a few hours,100%,82%,t,https://a0.muscache.com/im/users/14942/profile...,https://a0.muscache.com/im/users/14942/profile...,Wallingford,5.0,5.0,"['email', 'phone']",t,t,Neighborhood highlights,Wallingford,Other neighborhoods,47.65444,-122.33629,Entire guesthouse,Entire home/apt,1,1.0,1 bath,1.0,1.0,"[""Free parking on premises"", ""Dedicated worksp...",$99.00,30,1125,30.0,30.0,1125.0,1125.0,30.0,1125.0,NaN,t,27,57,87,177,2025-09-25,161,0,0,95,1,0,0.0,2009-07-17,2024-09-07,4.60,4.67,4.67,4.83,4.77,4.88,4.57,str-opli-19-002622,f,3,3,0,0,0.82
1,9419,https://www.airbnb.com/rooms/9419,20250925032813,2025-09-25,city scrape,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"Lots of restaurants (see our guide book) bars,...",https://a0.muscache.com/pictures/56645186/e5fb...,30559,https://www.airbnb.com/users/show/30559,Angielena,2009-08-09,"Seattle, WA",I am a visual artist who is the director ...,within an hour,100%,97%,t,https://a0.muscache.com/im/pictures/user/User-...,https://a0.muscache.com/im/pictures/user/User-...,Georgetown,10.0,11.0,"['email', 'phone']",t,t,Neighborhood highlights,Georgetown,Other neighborhoods,47.55017,-122.31937,Private room in rental unit,Private room,2,3.0,3 shared baths,1.0,2.0,"[""Baking sheet"", ""Luggage dropoff allowed"", ""C...",$71.00,2,90,2.0,2.0,90.0,90.0,2.0,90.0,NaN,t,0,22,52,327,2025-09-25,220,14,1,60,14,84,5964.0,2010-07-30,2025-08-31,4.73,4.80,4.75,4.92,4.89,4.70,4.69,Exempt,f,10,0,10,0,1.19
2,9596,https://www.airbnb.com/rooms/9596,20250925032813,2025-09-25,previous scrape,"the down home , spacious, central and fab!","We are in a great neighborhood, quiet, full of...","if you arrive early for check in at 3, I reco...",https://a0.muscache.com/pictures/665252/102d18...,14942,https://www.airbnb.com/users/show/14942,Joyce,2009-04-26,"Seattle, WA",I am a therapist/innkeeper.I know my city well...,within a few hours,100%,82%,t,https://a0.muscache.com/im/users/14942/profile...,https://a0.muscache.com/im/users/14942/profile...,Wallingford,5.0,5.0,"['email', 'phone']",t,t,Neighborhood highlights,Wallingford,Other neighborhoods,47.65608,-122.33602,Entire rental unit,Entire home/apt,4,

## 2. Restrict to Seattle Neighborhoods

To make sure the dataset is limited to Seattle listings, keep only observations where `neighbourhood_cleansed` appears in the Seattle neighborhood reference file.


In [2]:
valid_neighborhoods = neighborhoods["neighbourhood"].dropna().unique()

listings_cleaned = listings[
    listings["neighbourhood_cleansed"].isin(valid_neighborhoods)
].copy()

print(f"Rows before Seattle neighborhood filter: {len(listings):,}")
print(f"Rows after Seattle neighborhood filter: {len(listings_cleaned):,}")

listings_cleaned.head()


Rows before Seattle neighborhood filter: 6,996
Rows after Seattle neighborhood filter: 6,996


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6606,https://www.airbnb.com/rooms/6606,20250925032813,2025-09-25,city scrape,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","A peaceful yet highly accessible neighborhood,...",https://a0.muscache.com/pictures/45742/21116d7...,14942,https://www.airbnb.com/users/show/14942,Joyce,2009-04-26,"Seattle, WA",I am a therapist/innkeeper.I know my city well...,within a few hours,100%,82%,t,https://a0.muscache.com/im/users/14942/profile...,https://a0.muscache.com/im/users/14942/profile...,Wallingford,5.0,5.0,"['email', 'phone']",t,t,Neighborhood highlights,Wallingford,Other neighborhoods,47.65444,-122.33629,Entire guesthouse,Entire home/apt,1,1.0,1 bath,1.0,1.0,"[""Free parking on premises"", ""Dedicated worksp...",$99.00,30,1125,30.0,30.0,1125.0,1125.0,30.0,1125.0,NaN,t,27,57,87,177,2025-09-25,161,0,0,95,1,0,0.0,2009-07-17,2024-09-07,4.60,4.67,4.67,4.83,4.77,4.88,4.57,str-opli-19-002622,f,3,3,0,0,0.82
1,9419,https://www.airbnb.com/rooms/9419,20250925032813,2025-09-25,city scrape,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"Lots of restaurants (see our guide book) bars,...",https://a0.muscache.com/pictures/56645186/e5fb...,30559,https://www.airbnb.com/users/show/30559,Angielena,2009-08-09,"Seattle, WA",I am a visual artist who is the director ...,within an hour,100%,97%,t,https://a0.muscache.com/im/pictures/user/User-...,https://a0.muscache.com/im/pictures/user/User-...,Georgetown,10.0,11.0,"['email', 'phone']",t,t,Neighborhood highlights,Georgetown,Other neighborhoods,47.55017,-122.31937,Private room in rental unit,Private room,2,3.0,3 shared baths,1.0,2.0,"[""Baking sheet"", ""Luggage dropoff allowed"", ""C...",$71.00,2,90,2.0,2.0,90.0,90.0,2.0,90.0,NaN,t,0,22,52,327,2025-09-25,220,14,1,60,14,84,5964.0,2010-07-30,2025-08-31,4.73,4.80,4.75,4.92,4.89,4.70,4.69,Exempt,f,10,0,10,0,1.19
2,9596,https://www.airbnb.com/rooms/9596,20250925032813,2025-09-25,previous scrape,"the down home , spacious, central and fab!","We are in a great neighborhood, quiet, full of...","if you arrive early for check in at 3, I reco...",https://a0.muscache.com/pictures/665252/102d18...,14942,https://www.airbnb.com/users/show/14942,Joyce,2009-04-26,"Seattle, WA",I am a therapist/innkeeper.I know my city well...,within a few hours,100%,82%,t,https://a0.muscache.com/im/users/14942/profile...,https://a0.muscache.com/im/users/14942/profile...,Wallingford,5.0,5.0,"['email', 'phone']",t,t,Neighborhood highlights,Wallingford,Other neighborhoods,47.65608,-122.33602,Entire rental unit,Entire home/apt,4,

## 3. Keep Only Columns Relevant for Pricing Analysis

The dropped fields are mostly URLs, long text descriptions, duplicate geography fields, host metadata, or administrative fields that are not needed for this pricing analysis.


In [3]:
# Keep only relevant columns for analysis
cols_to_keep = [
    "id",
    "neighbourhood_group_cleansed",
    "neighbourhood_cleansed",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "room_type",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating"
]

listings_cleaned = listings_cleaned[cols_to_keep].copy()

print(f"Shape after dropping unused columns: {listings_cleaned.shape}")


Shape after dropping unused columns: (6996, 12)


## 4. Clean Price Column


In [4]:
listings_cleaned["price"] = (
    listings_cleaned["price"]
    .astype(str)
    .str.replace(r"[$,]", "", regex=True)
)

listings_cleaned["price"] = pd.to_numeric(
    listings_cleaned["price"],
    errors="coerce"
)

## 5. Check Missing Values

Price is the most important variable for this analysis, so missing price values are handled directly in the next step.


In [5]:
missing_summary = (
    listings_cleaned.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_summary[missing_summary["missing_count"] > 0].head(25)


,missing_count
review_scores_rating,887
price,775
beds,711
bathrooms,708
bedrooms,151


## 6. Backfill Missing Prices from Prior Quarter Data

Some listings are missing prices in the main quarter. Because this appears likely to be a scraping or availability issue, missing prices are backfilled using the same listing's price from the prior quarter when available.

Backfilled prices preserve sample size, but they should be interpreted as approximate because they come from a previous quarter.


In [6]:
missing_price_before = listings_cleaned["price"].isna().sum()
print(f"Missing prices before backfill: {missing_price_before:,}")

prior_prices = listings_prior[["id", "price"]].copy()
prior_prices = prior_prices.rename(columns={"price": "prior_quarter_price"})

prior_prices["prior_quarter_price"] = (
    prior_prices["prior_quarter_price"]
    .astype(str)
    .str.replace(r"[$,]", "", regex=True)
)

prior_prices["prior_quarter_price"] = pd.to_numeric(
    prior_prices["prior_quarter_price"],
    errors="coerce"
)

listings_cleaned = listings_cleaned.merge(prior_prices, on="id", how="left")

listings_cleaned["price_backfilled"] = (
    listings_cleaned["price"].isna()
    & listings_cleaned["prior_quarter_price"].notna()
)

listings_cleaned["price"] = listings_cleaned["price"].fillna(
    listings_cleaned["prior_quarter_price"]
)

missing_price_after = listings_cleaned["price"].isna().sum()
backfilled_count = missing_price_before - missing_price_after

print(f"Prices backfilled from prior quarter: {backfilled_count:,}")
print(f"Missing prices after backfill: {missing_price_after:,}")


Missing prices before backfill: 775
Prices backfilled from prior quarter: 348
Missing prices after backfill: 427


## 7. Drop Remaining Missing Prices

Listings that still have missing prices after the backfill step are removed because price is required for the pricing analysis.


In [7]:
rows_before = len(listings_cleaned)
listings_cleaned = listings_cleaned[listings_cleaned["price"].notna()].copy()
rows_after = len(listings_cleaned)

print(f"Rows before dropping remaining missing prices: {rows_before:,}")
print(f"Rows after dropping remaining missing prices: {rows_after:,}")
print(f"Rows dropped: {rows_before - rows_after:,}")


Rows before dropping remaining missing prices: 6,996
Rows after dropping remaining missing prices: 6,569
Rows dropped: 427


## 8. Inspect and Trim Price Outliers

Airbnb prices are highly skewed, and a small number of very expensive listings can distort averages. To reduce the influence of extreme observations, prices above the 99th percentile are removed.


In [8]:
price_cap = listings_cleaned["price"].quantile(0.99)
rows_before = len(listings_cleaned)

listings_cleaned = listings_cleaned[listings_cleaned["price"] <= price_cap].copy()
rows_after = len(listings_cleaned)

print(f"99th percentile price cap: ${price_cap:,.2f}")
print(f"Rows before outlier trim: {rows_before:,}")
print(f"Rows after outlier trim: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")


99th percentile price cap: $1,731.40
Rows before outlier trim: 6,569
Rows after outlier trim: 6,503
Rows removed: 66


## 9. Filter to Comparable Short Term Listings

To make neighborhood comparisons more interpretable, the sample is restricted to typical short term stays and listings that accommodate at least two guests.

The final analysis sample keeps:

- listings with `minimum_nights <= 7`
- listings with `accommodates >= 2`


In [9]:
rows_before = len(listings_cleaned)

listings_cleaned = listings_cleaned[
    (listings_cleaned["minimum_nights"] <= 7)
    & (listings_cleaned["accommodates"] >= 2)
].copy()

rows_after = len(listings_cleaned)

print(f"Rows before short term comparability filter: {rows_before:,}")
print(f"Rows after short term comparability filter: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")


Rows before short term comparability filter: 6,503
Rows after short term comparability filter: 4,486
Rows removed: 2,017


## 10. Final Dataset Summary


In [10]:
listings_cleaned.head()

,id,neighbourhood_group_cleansed,neighbourhood_cleansed,price,accommodates,bedrooms,beds,bathrooms,room_type,minimum_nights,number_of_reviews,review_scores_rating,prior_quarter_price,price_backfilled
1,9419,Other neighborhoods,Georgetown,71.0,2,1.0,2.0,3.0,Private room,2,220,4.73,88.0,False
4,25002,Ballard,Whittier Heights,88.0,4,1.0,2.0,1.0,Entire home/apt,2,1139,4.93,96.0,False
6,119103,Other neighborhoods,Fremont,92.0,3,2.0,2.0,1.0,Entire home/apt,3,597,4.94,105.0,False
9,215882,Rainier Valley,Columbia City,111.0,2,1.0,1.0,1.0,Entire home/apt,2,419,4.89,NaN,False
11,226536,Magnolia,Lawton Park,74.0,2,1.0,1.0,1.0,Private room,1,417,4.83,70.0,False


In [11]:
print(f"Final dataset shape: {listings_cleaned.shape}")

listings_cleaned.describe().round(2)


Final dataset shape: (4486, 14)


,id,price,accommodates,bedrooms,beds,bathrooms,minimum_nights,number_of_reviews,review_scores_rating,prior_quarter_price
count,4.486000e+03,4486.00,4486.00,4440.00,4279.00,4280.00,4486.00,4486.00,4312.00,3995.00
mean,6.878515e+17,188.82,4.50,1.80,2.39,1.51,1.99,112.14,4.85,225.94
std,5.648143e+17,129.93,2.64,1.23,1.58,0.81,1.10,142.50,0.22,147.30
min,9.419000e+03,34.00,2.00,0.00,0.00,0.00,1.00,0.00,1.00,39.00
25%,4.505780e+07,114.00,2.00,1.00,1.00,1.00,1.00,20.00,4.81,132.50
50%,8.185394e+17,157.00,4.00,2.00,2.00,1.00,2.00,61.00,4.91,189.00
75%,1.192324e+18,222.00,6.00,2.00,3.00,2.00,2.00,146.00,4.97,277.00
max,1.516826e+18,1717.00,16.00,13.00,15.00,16.00,7.00,1577.00,5.00,1950.00


## 11. Export Cleaned Data

The cleaned dataset is exported for use in the analysis notebook. Raw data files are not included in the public repository because of file size.


In [12]:
listings_cleaned = listings_cleaned.drop(columns=["prior_quarter_price"], errors="ignore")

listings_cleaned.to_csv("../data/listings_cleaned.csv", index=False)
print("Cleaned file exported to ../data/listings_cleaned.csv")


Cleaned file exported to ../data/listings_cleaned.csv
